# ML Challenge 2026 — Business ER on Colab (profiled)

Runs the same pipeline as SageMaker/EC2, **one step at a time**, saving `work/` to Google Drive after
every step. If Colab disconnects, reconnect and **Run all** again: finished steps are skipped.

Every step is profiled; everything lands in `MyDrive/er2026/work/<RUN>/profile/`:
- `hardware.json` — cores, RAM, CPU model, settings
- `<step>_resources.csv` — every 5 s: busy cores, CPU of the process tree, RSS, free RAM, swap, disk IO
- `<step>_stdout.log` — pipeline log (includes its own sub-step timings)
- `pyspy_<step>.txt` — sampled call stacks (which functions eat the time, incl. numba/scipy native code)
- `summary.json` — per-step minutes, parallel efficiency, peak memory

**Before you start**
1. Runtime → Change runtime type. CPU High-RAM is enough; a GPU runtime only helps via its extra CPU cores/RAM.
2. In Google Drive create `MyDrive/er2026/` containing:
   - `business_entity_resolution/` (this code folder, as-is)
   - `dataset/train/*.tsv` and `dataset/test/*.tsv` (the 7 challenge files)
   - `validate_submission.py` (from `student_resource/utils/`)

In [ ]:
import os, psutil, platform
ram_gb = psutil.virtual_memory().total / 2**30
print(f"cores logical={os.cpu_count()} physical={psutil.cpu_count(logical=False)}  RAM={ram_gb:.0f} GB")
!lscpu | grep -E "Model name|Thread|Core|Socket|MHz|L3"
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU (fine: pipeline is CPU-bound)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/er2026'
RUN = 'colab-v1'                      # change to start a fresh run
D_WORK, D_OUT = f'{DRIVE}/work/{RUN}', f'{DRIVE}/output/{RUN}'
os.makedirs(D_WORK, exist_ok=True); os.makedirs(D_OUT, exist_ok=True)
for p in ['business_entity_resolution/run.py', 'dataset/train/train_source1.tsv', 'dataset/test/test_source1.tsv']:
    assert os.path.exists(f'{DRIVE}/{p}'), f'missing {DRIVE}/{p}'
print('Drive layout OK')

In [ ]:
# copy code + data to fast local disk; restore any finished work from Drive
!rm -rf /content/er && mkdir -p /content/er
!cp -r "$DRIVE/business_entity_resolution" /content/er/code
!rsync -a --info=progress2 "$DRIVE/dataset/" /content/er/data/
!rsync -a --exclude profile "$D_WORK/" /content/er/work/
!ls /content/er/work

In [ ]:
# isolated Python 3.11 env (pinned numba/numpy do not support Colab's newer Python)
!pip -q install uv py-spy
!uv venv -q --python 3.11 /content/venv
!uv pip install -q --python /content/venv/bin/python -r /content/er/code/requirements.txt
!/content/venv/bin/python -c "import numba, lightgbm, numpy; print('numba', numba.__version__, 'lgb', lightgbm.__version__)"

In [ ]:
# pick settings from available RAM (train_keep_s1 = share of training S1 used)
if ram_gb >= 80:   keep = 0.6
elif ram_gb >= 60: keep = 0.45
elif ram_gb >= 45: keep = 0.3    # tight: Colab has no swap
else:              keep = None
assert keep, f'{ram_gb:.0f} GB RAM is too small for the full run: enable High-RAM'
SETS = [f'n_jobs={os.cpu_count()}', 'chunk_rows=500', f'train_keep_s1={keep}']
print(SETS)

In [ ]:
# ---- profiling helpers: resource sampler (py-spy is attached per step below) --------------
import subprocess, time, threading, json, csv, shutil
PROF = f'{D_WORK}/profile'; os.makedirs(PROF, exist_ok=True)
cpu_model = [l for l in open('/proc/cpuinfo') if l.startswith('model name')][0].split(':')[1].strip()
HW = {'cores': os.cpu_count(), 'ram_gb': round(ram_gb, 1), 'cpu': cpu_model, 'sets': SETS}
json.dump(HW, open(f'{PROF}/hardware.json', 'w'), indent=1)
print(HW)


class Sampler(threading.Thread):
    # every `dt` s: CPU of the whole process tree, busy cores, RSS, free RAM, swap, disk IO
    def __init__(self, pid, path, dt=5):
        super().__init__(daemon=True)
        self.pid, self.path, self.dt = pid, path, dt
        self.stop, self.rows, self.seen = threading.Event(), [], {}

    def run(self):
        root = psutil.Process(self.pid); io0 = psutil.disk_io_counters(); t0 = time.time()
        psutil.cpu_percent(percpu=True)
        while not self.stop.wait(self.dt):
            try:
                procs = [root] + root.children(recursive=True)
            except psutil.NoSuchProcess:
                break
            rss = cpu = 0.0
            for q in procs:
                try:
                    q = self.seen.setdefault(q.pid, q)   # reuse objects so cpu_percent has a baseline
                    rss += q.memory_info().rss; cpu += q.cpu_percent()
                except psutil.Error:
                    pass
            per = psutil.cpu_percent(percpu=True); vm = psutil.virtual_memory(); io = psutil.disk_io_counters()
            r = {'t': round(time.time() - t0), 'n_procs': len(procs), 'tree_cpu_pct': round(cpu),
                 'busy_cores': sum(c > 80 for c in per), 'sys_cpu_pct': round(sum(per) / len(per)),
                 'rss_gb': round(rss / 2**30, 2), 'avail_gb': round(vm.available / 2**30, 2),
                 'swap_gb': round(psutil.swap_memory().used / 2**30, 2),
                 'read_mb': round((io.read_bytes - io0.read_bytes) / 2**20),
                 'write_mb': round((io.write_bytes - io0.write_bytes) / 2**20)}
            self.rows.append(r)
            if len(self.rows) % 12 == 0:   # console heartbeat once a minute
                print(f"   [mon t={r['t']}s] busy_cores={r['busy_cores']}/{HW['cores']} tree_cpu={r['tree_cpu_pct']}% "
                      f"rss={r['rss_gb']}GB avail={r['avail_gb']}GB swap={r['swap_gb']}GB procs={r['n_procs']}", flush=True)
        with open(self.path, 'w', newline='') as f:
            if self.rows:
                w = csv.DictWriter(f, fieldnames=list(self.rows[0])); w.writeheader(); w.writerows(self.rows)

    def summary(self):
        R = self.rows or [{}]
        n = max(len(self.rows), 1)
        col = lambda k: [r.get(k, 0) for r in R]
        return {'peak_rss_gb': max(col('rss_gb')), 'min_avail_gb': min(col('avail_gb')),
                'peak_swap_gb': max(col('swap_gb')), 'avg_busy_cores': round(sum(col('busy_cores')) / n, 1),
                'avg_tree_cpu_pct': round(sum(col('tree_cpu_pct')) / n),
                'share_time_single_core': round(sum(b <= 1 for b in col('busy_cores')) / n, 2),
                'disk_read_gb': round(R[-1].get('read_mb', 0) / 1024, 1),
                'disk_write_gb': round(R[-1].get('write_mb', 0) / 1024, 1)}

In [ ]:
# ---- run the pipeline step by step, profiled, checkpointed to Drive -------------------------
STEPS = ['prepare', 'block', 'rerank', 'features', 'train', 'predict']
env = dict(os.environ, NUMBA_CACHE_DIR='/tmp/numba', PYTHONUNBUFFERED='1')
PYSPY = shutil.which('py-spy')
timings = json.load(open(f'{PROF}/summary.json')) if os.path.exists(f'{PROF}/summary.json') else {}
for step in STEPS:
    marker = f'{D_WORK}/_done_{step}'
    if os.path.exists(marker):
        print(f'== {step}: already done, skipping'); continue
    t0 = time.time(); print(f'== {step}: running  ({time.strftime("%H:%M:%S")})', flush=True)
    cmd = ['/content/venv/bin/python', 'run.py', step, '--data', '/content/er/data', '--work', '/content/er/work',
           '--out', '/content/er/output', '--set', *SETS]
    if PYSPY:   # 20 Hz sampling of main + worker processes, incl. native (numba/scipy) frames
        cmd = [PYSPY, 'record', '--rate', '20', '--subprocesses', '--native',
               '--format', 'raw', '-o', f'/content/pyspy_{step}.txt', '--'] + cmd
    p = subprocess.Popen(cmd, cwd='/content/er/code', env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    mon = Sampler(p.pid, f'{PROF}/{step}_resources.csv'); mon.start()
    with open(f'{PROF}/{step}_stdout.log', 'w') as flog:
        for line in p.stdout:
            print(line, end=''); flog.write(line); flog.flush()
    rc = p.wait(); mon.stop.set(); mon.join()
    if os.path.exists(f'/content/pyspy_{step}.txt'):
        shutil.copy(f'/content/pyspy_{step}.txt', f'{PROF}/pyspy_{step}.txt')
    timings[step] = {'minutes': round((time.time() - t0) / 60, 1), 'rc': rc, **mon.summary()}
    json.dump(timings, open(f'{PROF}/summary.json', 'w'), indent=1)
    print(f'== {step}: {json.dumps(timings[step])}', flush=True)
    assert rc == 0, f'{step} failed (see log above)'
    subprocess.run(['rsync', '-a', '/content/er/work/', f'{D_WORK}/'], check=True)   # checkpoint
    open(marker, 'w').write(f'{time.time() - t0:.0f}s')
    print(f'== {step}: checkpoint saved to Drive', flush=True)

In [ ]:
# ---- profile report: time per step, parallel efficiency, top hot functions --------------------
import collections
print(f"{'step':9} {'min':>6} {'busy/cores':>11} {'1-core%':>8} {'peakRSS':>8} {'swap':>5}")
for k, v in timings.items():
    print(f"{k:9} {v['minutes']:6.1f} {v['avg_busy_cores']:5}/{HW['cores']:<5} "
          f"{100 * v['share_time_single_core']:7.0f}% {v['peak_rss_gb']:7.1f}G {v['peak_swap_gb']:5.1f}")
for step in timings:
    f = f'{PROF}/pyspy_{step}.txt'
    if not os.path.exists(f):
        continue
    self_t, tot = collections.Counter(), 0
    for line in open(f):
        stack, _, n = line.rstrip().rpartition(' ')
        if not n.isdigit():
            continue
        n = int(n); tot += n
        self_t[stack.split(';')[-1][:110]] += n
    print(f'\n--- {step}: top self-time frames ({tot} samples) ---')
    for fr, n in self_t.most_common(12):
        print(f'{100 * n / tot:5.1f}%  {fr}')

In [ ]:
!rsync -a /content/er/output/ "$D_OUT/"
!/content/venv/bin/python "$DRIVE/validate_submission.py" -m /content/er/output/matching_results.tsv -c /content/er/output/candidate_pairs.tsv -t /content/er/data/test
!cat /content/er/work/model/report.json 2>/dev/null | head -40